# N15 — Avaliação de sistemas multiagente

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a>
* Monitor: Alejandro Núñez Arroyo <a href="mailto:r299215@dac.unicamp.br">(r299215@dac.unicamp.br)</a>

Como saber se um agente funciona? Um agente é não determinístico, chama
ferramentas diferentes a cada execução e mexe em sistemas externos.

Vamos montar um sistema de reservas de viagem com três agentes (Researcher, Booker e Reviewer) no padrão
Subagents do LangChain e avaliá-lo de quatro formas, cada uma olhando um ponto distinto da execução:

| # | Avaliador | O que mede | Seção |
|---|---|---|---|
| A | Estado final | diff do banco antes/depois contra o estado esperado | §5 |
| B | Trajetória | sequência de tool-calls contra uma sequência de referência | §6 |
| C | Asserções | checagens determinísticas: data, orçamento, duplicatas, dano colateral | §7 |
| D | Juiz LLM binário | alucinações na mensagem final | §8 |

## 0.1 Pré-requisitos

- agente com tool calling e o loop de raciocínio;
- padrão multiagente (orquestrador + subagentes);
- Python: decoradores, `functools.wraps`, context managers;
- SQL básico (`INSERT`, `UPDATE`, `SELECT`);
- noção de teste automatizado.

As quatro técnicas são construídas aqui; só a B usa uma biblioteca externa (`agentevals`).

## 0.2 Objetivos da aula

Ao final deste notebook, você deverá saber:

1. Explicar por que a saída de texto de um agente é a evidência mais fraca sobre o que ele fez.
2. Instrumentar ferramentas para registrar a trajetória de chamadas.
3. Gerar um **golden end-state** executando uma solução de referência sobre um ambiente limpo.
4. Avaliar por estado final com hash e diff (`missing` / `unexpected`).
5. Comparar trajetórias nos quatro modos do `agentevals`.
6. Escrever asserções determinísticas a partir dos requisitos declarados de uma tarefa.
7. Montar um juiz LLM binário com saída estruturada e few-shot.
8. Distinguir os dois tipos de *ground truth* e o custo de cada um.
9. Reconhecer os falsos negativos de cada avaliador.
10. Projetar um dataset de avaliação que inclua o caso negativo.

## 0.3 Mapa do notebook

1. Por que avaliar um agente é difícil.
2. O ambiente: a base de dados.
3. Ferramentas instrumentadas.
4. Os três agentes e o harness de execução.
5. Padrão A - estado final.
6. Padrão B - trajetória e uso de ferramentas.
7. Padrão C - asserções determinísticas.
8. Padrão D - juiz LLM binário.
9. O dataset de tarefas.
10. A suíte completa.
11. Síntese: o que cada avaliador mede, com o quê e contra o quê.
12. Exercícios, resumo e referências.

## 0.4 Preparação do ambiente

Cinco pacotes: `langchain` e `langchain-ollama` (agentes e cliente do modelo), `langgraph` (runtime do
`create_agent`), `agentevals` (comparação de trajetórias, §6), `pandas` (tabelas de resultado) e
`python-dotenv` (leitura da chave).

O modelo é o `gpt-oss:120b`, servido pelo Ollama Cloud em `https://ollama.com`, o que dispensa instalar o
Ollama localmente e faz o notebook funcionar igual no Colab e em qualquer máquina.

Crie uma chave em [ollama.com/settings/keys](https://ollama.com/settings/keys) e deixe-a em
`OLLAMA_API_KEY`, seguindo qualquer uma das formas do Notebook 3 (arquivo `.env`, variável de ambiente do
sistema, ou a célula comentada abaixo). Não escreva a chave no código.


In [14]:
!pip install langchain langchain-ollama langgraph agentevals pandas python-dotenv



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
# # Se você não tiver a chave no Google Colab:
# import os
# os.environ["OLLAMA_API_KEY"] = ""

In [16]:
import copy, functools, hashlib, inspect, json, os, sqlite3, threading, time
from contextlib import contextmanager

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv()

judge_llm = init_chat_model(model="ollama:qwen3:14b", base_url="http://localhost:11500", reasoning=False)


# 1. Por que avaliar um agente é difícil

Testar uma função é comparar entrada e saída esperada. Um agente quebra essa simetria em três pontos:

- **não há saída única**: duas respostas corretas podem não ter uma palavra em comum, e uma errada pode
  estar bem redigida;
- **não há caminho único**: buscar o voo antes ou depois do hotel dá no mesmo, e exigir uma sequência exata
  transforma variação legítima em falha;
- **o efeito colateral importa mais que a resposta**: uma reserva a mais no banco é um erro real mesmo com
  a mensagem final perfeita.

A pergunta que organiza o notebook é onde procurar a evidência:

| Onde olhar | Avaliador | Vantagem | Ponto cego |
|---|---|---|---|
| Estado final do mundo | A | detecta qualquer desvio, inclusive os não previstos | não diz qual requisito quebrou |
| Sequência de tool-calls | B | mostra como o agente chegou lá | caminho alternativo válido reprova |
| Requisitos declarados | C | diagnóstico preciso e barato | só pega o que alguém pensou em assertar |
| Mensagem final | D | única que lê linguagem livre | não determinístico, custa uma chamada de LLM |

As sobreposições não são redundância: A e C olham o mesmo estado, mas A pega desvios imprevistos e C diz
qual requisito foi violado. B pode reprovar uma execução correta por caminho alternativo, e aprovar uma
errada, já que a chamada esperada não garante que a escrita chegou ao banco. D é o único não
determinístico, o que faz dele complemento aos outros três, nunca substituto.

# 2. O ambiente: a base de dados

Só dá para avaliar por estado se o estado for inspecionável. O ambiente é um SQLite em memória com duas
partes:

- catálogo (`flights`, `hotels`), somente leitura, que o Researcher consulta;
- estado mutável (`bookings` e `flights.seats`), que o Booker modifica e é a única coisa avaliada.

Três detalhes sustentam a avaliação:

- `snapshot()` serializa com ordem fixa e chaves ordenadas, então dois estados equivalentes produzem o
  mesmo hash (o τ-bench resolve isso com `consistent_hash(to_hashable(data))`);
- `create_booking` decrementa `flights.seats` além de inserir a linha, então uma reserva criada e depois
  cancelada deixa rastro mesmo com a tabela `bookings` limpa;
- `MUTABLE_TABLES` define o escopo da avaliação: só o que está na tupla entra no hash e no diff. Tabelas de
  log ou cache ficam de fora, senão qualquer execução divergiria por ruído.

In [17]:
SCHEMA = """
CREATE TABLE flights (
    id TEXT PRIMARY KEY, origin TEXT, destination TEXT, date TEXT,
    price REAL, airline TEXT, seats INTEGER
);
CREATE TABLE hotels (
    id TEXT PRIMARY KEY, city TEXT, name TEXT, stars INTEGER, price_per_night REAL
);
CREATE TABLE bookings (
    id TEXT PRIMARY KEY, user_id TEXT, kind TEXT, item_id TEXT,
    start_date TEXT, price REAL, status TEXT
);
"""

FLIGHTS = [
    ("FL-101", "MAD", "LIM", "2026-09-12",  980.0, "Iberia",     4),
    ("FL-102", "MAD", "LIM", "2026-09-12", 1240.0, "LATAM",      9),
    ("FL-103", "MAD", "LIM", "2026-09-13",  760.0, "Air France", 2),
    ("FL-201", "LIM", "MAD", "2026-09-20",  890.0, "Iberia",     6),
]
HOTELS = [
    ("HT-1", "LIM", "Miraflores Suites", 4, 120.0),
    ("HT-2", "LIM", "Barranco Hostal",   2,  45.0),
    ("HT-3", "LIM", "Surco Business",    3,  58.0),
]

# Tabelas que o agente pode modificar -> são as que entram na avaliação de estado.
MUTABLE_TABLES = ("bookings", "flights")


def fresh_db() -> sqlite3.Connection:
    """Cria um banco em memória com o catálogo semeado e ZERO reservas."""
    conn = sqlite3.connect(":memory:", check_same_thread=False)  # as tools rodam em thread pool
    conn.row_factory = sqlite3.Row
    conn.executescript(SCHEMA)
    conn.executemany("INSERT INTO flights VALUES (?,?,?,?,?,?,?)", FLIGHTS)
    conn.executemany("INSERT INTO hotels  VALUES (?,?,?,?,?)",     HOTELS)
    conn.commit()
    return conn


DB = fresh_db()


def snapshot(conn: sqlite3.Connection | None = None) -> dict:
    """Estado mutável em forma canônica (ordem determinística)."""
    conn = conn or DB
    return {
        t: [dict(r) for r in conn.execute(f"SELECT * FROM {t} ORDER BY id")]
        for t in MUTABLE_TABLES
    }


def state_hash(state: dict) -> str:
    """Hash estável de um snapshot. Mesmo estado -> mesmo hash, não importa o caminho."""
    return hashlib.sha256(json.dumps(state, sort_keys=True, default=str).encode()).hexdigest()[:16]


print("Estado inicial:", state_hash(snapshot()), "| reservas:", len(snapshot()["bookings"]))

Estado inicial: 30edc5bc2674738c | reservas: 0


`fresh_db()` roda antes de cada tarefa, então nenhuma herda sujeira da anterior. Sem esse isolamento, a
segunda tarefa passa a depender do que a primeira fez.

# 3. Ferramentas instrumentadas

A §6 precisa de um log de tudo o que foi chamado, e a §5 precisa reproduzir uma solução de referência sem
poluir esse log. Por isso cada ferramenta é registrada duas vezes:

1. `RAW[nome]`: a função pura, usada no replay da solução de referência;
2. `traced`: envolve a função e empilha cada chamada em `TRACE` com nome, argumentos e resultado.

O log é plano e atravessa os três subagentes: como toda ferramenta de baixo nível passa pelo mesmo
`traced`, o `TRACE` registra a trajetória do sistema inteiro, não importa qual subagente chamou o quê.

> Sem `functools.wraps` o LangChain infere o schema da ferramenta a partir da assinatura do wrapper
> (`*args, **kwargs`) e o tool calling falha.

In [18]:
TRACE: list[dict] = []
LOCK = threading.RLock()   # as tools rodam concorrentemente -> DB e TRACE precisam de lock
RAW: dict = {}             # nome -> função pura (sem trace), para o replay de referência


def traced(fn):
    """Instrumenta uma função-tool: registra (nome, args, resultado) em TRACE."""
    sig = inspect.signature(fn)

    @functools.wraps(fn)   # <- para que o LangChain infira o schema corretamente
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        with LOCK:
            out = fn(*args, **kwargs)
            TRACE.append({"name": fn.__name__, "args": dict(bound.arguments), "result": out})
        return out

    return wrapper


def as_tool(fn):
    """Registra a versão pura e devolve a tool do LangChain (instrumentada)."""
    RAW[fn.__name__] = fn
    return tool(traced(fn))


# --------------------------- catálogo (somente leitura) ---------------------------

def search_flights(origin: str, destination: str, date: str, max_price: float = 1e9) -> str:
    """Busca voos. origin/destination são códigos IATA (MAD, LIM). date no formato YYYY-MM-DD."""
    rows = DB.execute(
        "SELECT * FROM flights WHERE origin=? AND destination=? AND date=? AND price<=? ORDER BY price",
        (origin, destination, date, max_price),
    ).fetchall()
    return json.dumps([dict(r) for r in rows], ensure_ascii=False)


def search_hotels(city: str, max_price_per_night: float = 1e9) -> str:
    """Busca hotéis em uma cidade (código IATA) abaixo de um preço por noite."""
    rows = DB.execute(
        "SELECT * FROM hotels WHERE city=? AND price_per_night<=? ORDER BY price_per_night",
        (city, max_price_per_night),
    ).fetchall()
    return json.dumps([dict(r) for r in rows], ensure_ascii=False)


# --------------------------- estado (escrita) ---------------------------

def create_booking(user_id: str, kind: str, item_id: str, start_date: str, price: float) -> str:
    """Cria uma reserva REAL no banco de dados. kind é 'flight' ou 'hotel'. Devolve o booking_id."""
    if kind == "flight":
        row = DB.execute("SELECT * FROM flights WHERE id=?", (item_id,)).fetchone()
        if row is None:
            return json.dumps({"error": "flight_not_found", "item_id": item_id})
        if row["seats"] <= 0:
            return json.dumps({"error": "no_seats_available", "item_id": item_id})
        DB.execute("UPDATE flights SET seats = seats - 1 WHERE id=?", (item_id,))
    elif kind == "hotel":
        if DB.execute("SELECT 1 FROM hotels WHERE id=?", (item_id,)).fetchone() is None:
            return json.dumps({"error": "hotel_not_found", "item_id": item_id})
    else:
        return json.dumps({"error": "invalid_kind", "kind": kind})

    n = DB.execute("SELECT COUNT(*) AS c FROM bookings").fetchone()["c"]
    booking_id = f"BK-{n + 1:04d}"
    DB.execute(
        "INSERT INTO bookings VALUES (?,?,?,?,?,?,?)",
        (booking_id, user_id, kind, item_id, start_date, price, "confirmed"),
    )
    DB.commit()
    return json.dumps({"booking_id": booking_id, "status": "confirmed"}, ensure_ascii=False)


def get_booking(booking_id: str) -> str:
    """Consulta uma reserva específica pelo seu id (por exemplo BK-0001)."""
    row = DB.execute("SELECT * FROM bookings WHERE id=?", (booking_id,)).fetchone()
    return json.dumps(dict(row) if row else {"error": "not_found"}, ensure_ascii=False)


def list_bookings(user_id: str) -> str:
    """Lista todas as reservas de um usuário."""
    rows = DB.execute("SELECT * FROM bookings WHERE user_id=? ORDER BY id", (user_id,)).fetchall()
    return json.dumps([dict(r) for r in rows], ensure_ascii=False)


def cancel_booking(booking_id: str) -> str:
    """Cancela uma reserva existente (status -> 'cancelled') e libera o assento se for um voo."""
    row = DB.execute("SELECT * FROM bookings WHERE id=?", (booking_id,)).fetchone()
    if row is None:
        return json.dumps({"error": "not_found"})
    DB.execute("UPDATE bookings SET status='cancelled' WHERE id=?", (booking_id,))
    if row["kind"] == "flight":
        DB.execute("UPDATE flights SET seats = seats + 1 WHERE id=?", (row["item_id"],))
    DB.commit()
    return json.dumps({"booking_id": booking_id, "status": "cancelled"})


T_SEARCH_FLIGHTS = as_tool(search_flights)
T_SEARCH_HOTELS  = as_tool(search_hotels)
T_CREATE_BOOKING = as_tool(create_booking)
T_GET_BOOKING    = as_tool(get_booking)
T_LIST_BOOKINGS  = as_tool(list_bookings)
T_CANCEL_BOOKING = as_tool(cancel_booking)

print("tools registradas:", list(RAW))

tools registradas: ['search_flights', 'search_hotels', 'create_booking', 'get_booking', 'list_bookings', 'cancel_booking']


A assimetria de permissões já está nas ferramentas: `search_*` só lê, `create_booking` e `cancel_booking`
escrevem, `get_booking` e `list_bookings` leem o estado mutável. Na próxima seção cada subagente recebe
apenas um subconjunto.

`create_booking` devolve erro como dado, não como exceção (`{"error": "flight_not_found"}`) — a mesma
distinção do MCP entre `isError` e erro de protocolo: um erro de execução vira contexto para o modelo se
corrigir; uma exceção mataria a execução.

# 4. Os três agentes

Cada subagente é um `create_agent` comum, envolvido num `@tool` que o orquestrador invoca. O orquestrador
não enxerga as ferramentas de baixo nível, só três colegas para quem delegar.

```
                 ┌──────────────┐
   usuário ────► │ Orquestrador │
                 └──────┬───────┘
        research_travel │ book_travel │ review_booking
          ┌─────────────┼─────────────┐
          ▼             ▼             ▼
    ┌──────────┐  ┌──────────┐  ┌──────────┐
    │Researcher│  │  Booker  │  │ Reviewer │
    │ (RO)     │  │  (RW)    │  │ (RO+fix) │
    └──────────┘  └──────────┘  └──────────┘
          └─────── SQLite ──────────┘
```

As permissões são deliberadamente desiguais:

| Subagente | Ferramentas | Pode escrever? |
|---|---|---|
| Researcher | `search_flights`, `search_hotels` | não |
| Booker | `create_booking`, `get_booking` | sim |
| Reviewer | `get_booking`, `list_bookings`, `cancel_booking` | só para corrigir |

Se o Researcher alucinar um voo inexistente, não há caminho até a tabela: ele não tem `create_booking`. O
dano fica contido no texto. Restringir ferramentas costuma ser mais eficaz do que pedir bom comportamento
no prompt.

In [19]:
researcher = create_agent(
    model=llm,
    tools=[T_SEARCH_FLIGHTS, T_SEARCH_HOTELS],
    system_prompt=(
        "Você é um pesquisador de viagens. Use as ferramentas de busca para encontrar opções reais.\n"
        "Devolva uma lista compacta com id, data e preço de cada opção, e recomende a mais barata "
        "que caiba no orçamento.\n"
        "NUNCA reserve nada. NUNCA invente ids nem preços: use apenas os que as ferramentas devolverem.\n"
        "Se não houver resultados, diga explicitamente: 'SEM DISPONIBILIDADE'.\n"
        "Responda sempre em português."
    ),
)

booker = create_agent(
    model=llm,
    tools=[T_CREATE_BOOKING, T_GET_BOOKING],
    system_prompt=(
        "Você é o agente de reservas. Você executa a reserva EXATA que lhe pedirem com create_booking "
        "e depois a confirma lendo-a com get_booking.\n"
        "Regras: uma única reserva por item pedido; não reserve nada que não tenham pedido "
        "explicitamente; se create_booking devolver um erro, informe o erro e NÃO tente de novo com "
        "outro item.\n"
        "Devolva o booking_id e os dados salvos. Responda sempre em português."
    ),
)

reviewer = create_agent(
    model=llm,
    tools=[T_GET_BOOKING, T_LIST_BOOKINGS, T_CANCEL_BOOKING],
    system_prompt=(
        "Você é o revisor de qualidade. Você verifica contra o banco de dados (não contra o que lhe "
        "contarem) que as reservas do usuário coincidem com o que foi pedido: item, data, preço e que "
        "não haja duplicatas.\n"
        "Se encontrar uma reserva duplicada ou incorreta, cancele-a com cancel_booking e explique por "
        "quê.\n"
        "Responda 'VERIFICADO OK' se estiver tudo certo, ou descreva o problema. "
        "Responda sempre em português."
    ),
)


@tool("research_travel", description=(
    "Busca opções de voo e/ou hotel. Passe origem, destino, datas e orçamento em texto natural. "
    "Devolve as opções disponíveis. Não reserva nada."))
def research_travel(query: str) -> str:
    return researcher.invoke({"messages": [{"role": "user", "content": query}]})["messages"][-1].content


@tool("book_travel", description=(
    "Executa a reserva no banco de dados. Você DEVE passar user_id, kind ('flight'/'hotel'), "
    "item_id exato, data YYYY-MM-DD e preço."))
def book_travel(query: str) -> str:
    return booker.invoke({"messages": [{"role": "user", "content": query}]})["messages"][-1].content


@tool("review_booking", description=(
    "Verifica reservas já criadas contra os requisitos do usuário. Passe o user_id e o que foi pedido."))
def review_booking(query: str) -> str:
    return reviewer.invoke({"messages": [{"role": "user", "content": query}]})["messages"][-1].content


ORCHESTRATOR_PROMPT = (
    "Você coordena três subagentes para gerenciar viagens. Fluxo obrigatório:\n"
    "1) research_travel -> encontra opções reais.\n"
    "2) book_travel -> reserva a opção mais barata que cumpra TODOS os requisitos do usuário.\n"
    "3) review_booking -> verifica o resultado contra o que foi pedido.\n\n"
    "Regras duras:\n"
    "- Passe SEMPRE dados exatos a cada subagente: user_id, item_id, data e preço literais.\n"
    "- Se o usuário pedir VÁRIOS itens (por exemplo voo E hotel), faça uma chamada a book_travel para "
    "CADA item e não termine até ter reservado todos.\n"
    "- Se o Researcher disser SEM DISPONIBILIDADE, NÃO chame book_travel. Informe o usuário e "
    "termine.\n"
    "- Não reserve nada que o usuário não tenha pedido.\n"
    "- Termine SEMPRE com uma mensagem de texto ao usuário resumindo o que foi reservado (ou por que "
    "não).\n"
    "- Na mensagem final NÃO invente políticas de cancelamento, bagagem, assentos nem reembolsos: "
    "mencione apenas dados que tenham saído das ferramentas.\n"
    "Responda sempre em português."
)

orchestrator = create_agent(model=llm, tools=[research_travel, book_travel, review_booking],
                            system_prompt=ORCHESTRATOR_PROMPT)

## 4.1 O harness de execução

`run_task` parte de um banco limpo, tira o snapshot antes, roda o agente e devolve o snapshot posterior
junto com a trajetória e a mensagem final. Esse par antes/depois alimenta §5 e §7.

Duas decisões:

- **um crash é resultado válido**: o `try/except` devolve `error` preenchido em vez de interromper a suíte,
  então um agente que estoura o `recursion_limit` pontua como falha sem travar as demais tarefas;
- **`last_text` não pega `messages[-1]`**: o `gpt-oss` às vezes encerra com um `AIMessage` de `content`
  vazio (o output foi todo para `reasoning`), e o juiz da §8 acabaria avaliando string vazia.

In [20]:
@contextmanager
def sandbox_db():
    """Isola DB e TRACE globais (para o replay de referência sem sujar a execução real)."""
    global DB, TRACE
    old_db, old_trace = DB, TRACE
    DB, TRACE = fresh_db(), []
    try:
        yield DB
    finally:
        DB, TRACE = old_db, old_trace


def last_text(messages) -> str:
    """Última mensagem do assistente com texto de verdade.

    O gpt-oss às vezes encerra com um AIMessage de content vazio (todo o output foi para 'reasoning').
    Pegar cegamente messages[-1].content deixa o juiz do padrão D avaliando uma string vazia,
    então recuamos até encontrar conteúdo.
    """
    for msg in reversed(messages):
        if type(msg).__name__ != "AIMessage":
            continue
        content = msg.content
        if isinstance(content, list):             # blocos de conteúdo -> concatenar os de texto
            content = " ".join(b.get("text", "") for b in content if isinstance(b, dict))
        if content and content.strip():
            return content.strip()
    return ""


def run_task(agent, prompt: str, recursion_limit: int = 50) -> dict:
    """Executa o agente sobre um banco limpo e captura tudo o que é preciso para avaliá-lo."""
    global DB, TRACE
    DB, TRACE = fresh_db(), []

    before = snapshot()
    t0 = time.time()
    error = None
    final = ""
    messages = []
    try:
        res = agent.invoke({"messages": [{"role": "user", "content": prompt}]},
                           {"recursion_limit": recursion_limit})
        messages = res["messages"]
        final = last_text(messages)
    except Exception as exc:                      # um crash é um resultado válido: avalia-se como falha
        error = f"{type(exc).__name__}: {exc}"

    return {
        "before": before,
        "after": snapshot(),
        "trace": copy.deepcopy(TRACE),
        "final": final,
        "messages": messages,
        "error": error,
        "secs": round(time.time() - t0, 1),
    }

## 4.2 Uma execução de exemplo

Antes de avaliar, vamos ver o que o sistema faz com um pedido típico. As três saídas abaixo são as fontes
de evidência das próximas seções: a trajetória, o estado final e a mensagem ao usuário.

In [21]:
demo = run_task(orchestrator, (
    "Sou o usuário u-42. Quero voar de Madri (MAD) para Lima (LIM) no dia 2026-09-12, "
    "com um orçamento máximo de 1000 EUR. Reserve para mim."
))

print(f"⏱  {demo['secs']}s | erro: {demo['error']}\n")
print("TRAJETÓRIA (tool-calls através dos 3 subagentes):")
for i, step in enumerate(demo["trace"], 1):
    print(f"  {i}. {step['name']}({json.dumps(step['args'], ensure_ascii=False)})")

print("\nESTADO FINAL (tabela bookings):")
display(pd.DataFrame(demo["after"]["bookings"]))

print("\nMENSAGEM FINAL AO USUÁRIO:")
print(demo["final"])

⏱  13.3s | erro: None

TRAJETÓRIA (tool-calls através dos 3 subagentes):
  1. search_flights({"origin": "MAD", "destination": "LIM", "date": "2026-09-12", "max_price": 1000.0})
  2. create_booking({"user_id": "u-42", "kind": "flight", "item_id": "FL-101", "start_date": "2026-09-12", "price": 980.0})

ESTADO FINAL (tabela bookings):


,id,user_id,kind,item_id,start_date,price,status
0,BK-0001,u-42,flight,FL-101,2026-09-12,980.0,confirmed



MENSAGEM FINAL AO USUÁRIO:
A reserva do voo foi concluída com sucesso. Confira os detalhes:

- **Reserva ID:** BK-0001  
- **Usuário:** u-42  
- **Tipo:** Voo  
- **Voo:** FL-101  
- **Data:** 2026-09-12  
- **Preço:** 980,00 EUR  

Se precisar de ajuda com algo mais, estou à disposição! 😊


A mensagem final é convincente e é a evidência mais fraca das três: seria idêntica se `create_booking`
tivesse devolvido um `booking_id` sem que a transação chegasse à tabela.

A trajetória mostra mais, mas repare no que ficou de fora:

- `search_flights` aparece duas vezes com argumentos idênticos. Trabalho repetido que só a trajetória
  registra;
- não há `list_bookings` nem `cancel_booking`, e o único `get_booking` é o que o Booker faz para confirmar
  a própria escrita. Ou seja, algum subagente não cumpriu seu prompt — voltamos a isso na §6.

De todo modo, é a tabela `bookings` que desmentiria a mensagem final, e é por ela que começamos.

# 5. Padrão A - Estado final

Comparar o banco ao terminar contra o estado esperado, sem ler o transcript, nas duas direções: aconteceu
tudo o que se esperava, e não aconteceu mais nada.

O estado esperado não é escrito à mão. `build_golden()` executa uma solução de referência sobre um banco
limpo, então qualquer caminho diferente mas equivalente pontua igual.

São reportados dois níveis:

- **hash**: booleano, o estado bate ou não;
- **diff dirigido**: o que falta (`missing`) e o que sobra (`unexpected`), por tabela.

`unexpected` é o que detecta dano colateral — reservas duplicadas, assentos consumidos a mais, linhas
canceladas que ficaram para trás. Nada disso apareceria num teste que só checa se a reserva pedida existe.

> `_rows_content(rows, drop=("id",))` faz o diff ignorar o `id` autogerado. Sem isso, uma reserva correta
> gravada como `BK-0002` em vez de `BK-0001` apareceria em `missing` e em `unexpected` ao mesmo tempo.

In [22]:
def build_golden(reference_calls: list[tuple[str, dict]]) -> dict:
    """Gera o golden state replicando a solução de referência sobre um banco limpo."""
    with sandbox_db() as conn:
        for name, args in reference_calls:
            RAW[name](**args)          # função pura: não suja o TRACE
        return snapshot(conn)


def _rows_content(rows: list[dict], drop=("id",)) -> list[str]:
    """Linhas como conteúdo canônico, ignorando o id autogerado (para o diff diagnóstico)."""
    return sorted(json.dumps({k: v for k, v in r.items() if k not in drop}, sort_keys=True, default=str)
                  for r in rows)


def outcome_grade(after: dict, golden: dict) -> dict:
    """Padrão A: compara o estado final real contra o golden end-state, nas duas direções."""
    h_actual, h_golden = state_hash(after), state_hash(golden)

    missing, unexpected = {}, {}
    for table in MUTABLE_TABLES:
        got  = _rows_content(after[table])
        want = _rows_content(golden[table])
        got_pool = list(got)
        miss = []
        for row in want:                       # o esperado que NÃO aconteceu
            if row in got_pool:
                got_pool.remove(row)
            else:
                miss.append(row)
        if miss:
            missing[table] = miss
        if got_pool:                           # o que aconteceu e NÃO era esperado = dano colateral
            unexpected[table] = got_pool

    return {
        "score": h_actual == h_golden,         # binário, estilo τ-bench
        "hash_actual": h_actual,
        "hash_golden": h_golden,
        "missing": missing,
        "unexpected": unexpected,
    }


def diff_state(before: dict, after: dict) -> dict:
    """Diff legível antes/depois: quais linhas foram adicionadas ou alteradas."""
    out = {}
    for table in MUTABLE_TABLES:
        b = {r["id"]: r for r in before[table]}
        a = {r["id"]: r for r in after[table]}
        added   = [a[k] for k in a if k not in b]
        removed = [b[k] for k in b if k not in a]
        changed = [{"id": k, "antes": b[k], "depois": a[k]} for k in a if k in b and a[k] != b[k]]
        if added or removed or changed:
            out[table] = {"added": added, "removed": removed, "changed": changed}
    return out


# --- demo sobre a execução anterior ---
REF_DEMO = [("search_flights", {"origin": "MAD", "destination": "LIM", "date": "2026-09-12",
                                "max_price": 1000.0}),
            ("create_booking", {"user_id": "u-42", "kind": "flight", "item_id": "FL-101",
                                "start_date": "2026-09-12", "price": 980.0}),
            ("get_booking",    {"booking_id": "BK-0001"})]

golden_demo = build_golden(REF_DEMO)
print("DIFF DO ESTADO (o que o agente realmente mudou):")
print(json.dumps(diff_state(demo["before"], demo["after"]), indent=2, ensure_ascii=False))
print("\nAVALIAÇÃO vs GOLDEN:")
print(json.dumps(outcome_grade(demo["after"], golden_demo), indent=2, ensure_ascii=False))

DIFF DO ESTADO (o que o agente realmente mudou):
{
  "bookings": {
    "added": [
      {
        "id": "BK-0001",
        "user_id": "u-42",
        "kind": "flight",
        "item_id": "FL-101",
        "start_date": "2026-09-12",
        "price": 980.0,
        "status": "confirmed"
      }
    ],
    "removed": [],
    "changed": []
  },
  "flights": {
    "added": [],
    "removed": [],
    "changed": [
      {
        "id": "FL-101",
        "antes": {
          "id": "FL-101",
          "origin": "MAD",
          "destination": "LIM",
          "date": "2026-09-12",
          "price": 980.0,
          "airline": "Iberia",
          "seats": 4
        },
        "depois": {
          "id": "FL-101",
          "origin": "MAD",
          "destination": "LIM",
          "date": "2026-09-12",
          "price": 980.0,
          "airline": "Iberia",
          "seats": 3
        }
      }
    ]
  }
}

AVALIAÇÃO vs GOLDEN:
{
  "score": true,
  "hash_actual": "c7e062ad43f45172",
  "hash_

A saída acima admite duas leituras. Hash igual: o mundo terminou onde deveria. Hash diferente com diff
vazio: é o falso FAIL do padrão A — o hash inclui o `id` e o diff não, então é o mesmo resultado com ordem
de escrita diferente.

O caso que interessa é hash diferente com `unexpected` preenchido: houve algo no banco que ninguém pediu, e
é o tipo de erro que nenhuma asserção escrita à mão teria antecipado.

# 6. Padrão B - Trajetória e uso de ferramentas

A pergunta muda de *onde o agente chegou* para *como ele chegou*. Comparamos a sequência de tool-calls
contra uma de referência com a [`agentevals`](https://github.com/langchain-ai/agentevals), que espera
mensagens no formato OpenAI — daí a conversão do `TRACE`.

Escolher o modo de comparação é a decisão de projeto:

| Modo | Passa se | Quando usar |
|---|---|---|
| `strict` | mesma sequência, mesma ordem, sem extras | fluxos rígidos e auditáveis |
| `unordered` | mesmas chamadas, ordem livre | quando a ordem não é semântica |
| `subset` | o agente fez no máximo o que a referência faz | detectar passos a mais |
| `superset` | o agente fez pelo menos o que a referência faz | tolerar verificações extras |

A segunda decisão é quais argumentos precisam bater. `ARGS_OVERRIDES` exige igualdade exata nas escritas
(`create_booking`, `cancel_booking`) e ignora os argumentos das buscas: buscar com `max_price=1000` ou sem
filtro é indiferente, gravar `price=980` ou `price=1240` não.

In [23]:
from agentevals.trajectory.match import create_trajectory_match_evaluator


def trace_to_messages(trace: list) -> list[dict]:
    """TRACE plano -> mensagens estilo OpenAI que a agentevals entende."""
    msgs = []
    for step in trace:
        msgs.append({"role": "assistant", "tool_calls": [
            {"function": {"name": step["name"],
                          "arguments": json.dumps(step.get("args", {}), sort_keys=True, default=str)}}
        ]})
        msgs.append({"role": "tool", "content": str(step.get("result", ""))})
    return msgs


def reference_to_messages(reference_calls: list[tuple[str, dict]]) -> list[dict]:
    return trace_to_messages([{"name": n, "args": a, "result": ""} for n, a in reference_calls])


# A escrita precisa ser exata; as leituras/buscas são livres.
ARGS_OVERRIDES = {
    "create_booking":  "exact",
    "cancel_booking":  "exact",
    "search_flights":  "ignore",
    "search_hotels":   "ignore",
    "get_booking":     "ignore",
    "list_bookings":   "ignore",
}

EVALUATORS = {
    mode: create_trajectory_match_evaluator(
        trajectory_match_mode=mode,
        tool_args_match_mode="exact",
        tool_args_match_overrides=ARGS_OVERRIDES,
    )
    for mode in ("strict", "unordered", "subset", "superset")
}


def trajectory_grade(trace: list, reference_calls: list) -> dict:
    out_msgs = trace_to_messages(trace)
    ref_msgs = reference_to_messages(reference_calls)
    return {mode: bool(ev(outputs=out_msgs, reference_outputs=ref_msgs)["score"])
            for mode, ev in EVALUATORS.items()}


traj_demo = trajectory_grade(demo["trace"], REF_DEMO)
print("referência:", [n for n, _ in REF_DEMO])
print("agente    :", [s["name"] for s in demo["trace"]])
print("\n", json.dumps(traj_demo, indent=2))
print("\nLeitura: se 'superset' passa mas 'strict' não, o agente fez TUDO o que era necessário mais")
print("passos extras (tipicamente verificações redundantes do Reviewer). Isso é aceitável; omitir um")
print("passo não é.")

referência: ['search_flights', 'create_booking', 'get_booking']
agente    : ['search_flights', 'create_booking']

 {
  "strict": false,
  "unordered": false,
  "subset": true,
  "superset": false
}

Leitura: se 'superset' passa mas 'strict' não, o agente fez TUDO o que era necessário mais
passos extras (tipicamente verificações redundantes do Reviewer). Isso é aceitável; omitir um
passo não é.


O resultado mais comum é `superset=True` com `strict=False`: o agente fez tudo o que era necessário e mais
alguma coisa. Aqui o excedente foi a busca duplicada da §4.2, que também derruba `subset` e `unordered`.
Passos a mais são aceitáveis; omitir um passo não.

É aí que este padrão mostra seu valor: a busca repetida não aparece em nenhum outro avaliador. O estado
final é idêntico ao de uma execução limpa, os oito checks passam e o juiz não tem como saber. Se a
preocupação for custo ou latência, a trajetória é o único lugar onde isso fica visível.

Duas limitações explicam por que ele nunca é usado sozinho:

- **não prova efeito**: `create_booking` no log significa que a função foi chamada, não que a linha ficou
  no banco. `strict=True` com estado errado é possível;
- **não vê o que não foi instrumentado**: só passa pelo `TRACE` o que foi registrado com `as_tool()`, ou
  seja, as seis ferramentas de baixo nível. As três tools de delegação (`research_travel`, `book_travel`,
  `review_booking`) usam `@tool` puro e são invisíveis.

A consequência é concreta: o `ORCHESTRATOR_PROMPT` exige `review_booking` como passo 3 obrigatório e nenhum
dos quatro avaliadores consegue dizer se ele rodou. Foi o que aconteceu na §4.2: suíte verde e uma regra do
prompt possivelmente não cumprida.

O escopo do trace é uma decisão de avaliação. Se a pergunta é se o agente escreveu certo, as tools de baixo
nível bastam; se é se o orquestrador seguiu o fluxo, é preciso instrumentar a delegação (Exercício 7).

# 7. Padrão C - Asserções determinísticas

Checagens binárias que codificam os requisitos do usuário e as invariantes do sistema. Não precisam de
solução de referência e custam microssegundos, então podem rodar em toda execução.

Cada uma corresponde a um modo de falha concreto:

| Check | Modo de falha que pega |
|---|---|
| `n_bookings_exact` | reservou a mais (duplicata) ou a menos |
| `no_duplicates` | mesmo item reservado duas vezes por uma retentativa |
| `dates_match` | reservou o dia errado |
| `within_budget` | ignorou o orçamento e pegou a opção cara |
| `all_confirmed` | deixou reservas em `cancelled`, lixo de uma retentativa |
| `user_id_correct` | escreveu na conta errada |
| `items_in_catalog` | alucinou um `item_id` inexistente |
| `no_catalog_damage` | consumiu assentos que não correspondiam |

`TGC` (Task Goal Completion) é o agregado e passa se todos passarem: é o número que se reporta, e os oito
booleanos são o que se lê quando ele dá falso.

O ground truth aqui não é uma solução resolvida, e sim os requisitos declarados na tarefa: `user_id`,
`allowed_dates`, `budget` e `expect`. Ninguém precisou resolver a tarefa para escrever isso, e por isso
este padrão sobrevive em domínios sem resposta canônica.

In [24]:
def assertion_grade(task: dict, before: dict, after: dict) -> dict:
    """Padrão C: checagens determinísticas derivadas dos requisitos do usuário."""
    bookings = after["bookings"]
    expected = task["expect"]                       # lista de reservas esperadas
    catalog_ids = {f[0] for f in FLIGHTS} | {h[0] for h in HOTELS}

    seats_before = {r["id"]: r["seats"] for r in before["flights"]}
    seats_after  = {r["id"]: r["seats"] for r in after["flights"]}
    expected_seat_drop = {}
    for e in expected:
        if e["kind"] == "flight":
            expected_seat_drop[e["item_id"]] = expected_seat_drop.get(e["item_id"], 0) + 1

    checks = {
        "n_bookings_exact": len(bookings) == len(expected),
        "no_duplicates": len({(b["user_id"], b["kind"], b["item_id"]) for b in bookings}) == len(bookings),
        "all_confirmed": all(b["status"] == "confirmed" for b in bookings),
        "user_id_correct": all(b["user_id"] == task["user_id"] for b in bookings),
        "dates_match": all(b["start_date"] in task["allowed_dates"] for b in bookings),
        "within_budget": all(b["price"] <= task["budget"].get(b["kind"], 0) for b in bookings),
        "items_in_catalog": all(b["item_id"] in catalog_ids for b in bookings),
        "no_catalog_damage": all(
            seats_before[fid] - seats_after[fid] == expected_seat_drop.get(fid, 0)
            for fid in seats_before
        ),
    }
    checks["TGC"] = all(checks.values())            # Task Goal Completion
    return checks


print(json.dumps(assertion_grade(
    {"user_id": "u-42", "allowed_dates": ["2026-09-12"], "budget": {"flight": 1000.0},
     "expect": [{"kind": "flight", "item_id": "FL-101"}]},
    demo["before"], demo["after"]), indent=2))

{
  "n_bookings_exact": true,
  "no_duplicates": true,
  "all_confirmed": true,
  "user_id_correct": true,
  "dates_match": true,
  "within_budget": true,
  "items_in_catalog": true,
  "no_catalog_damage": true,
  "TGC": true
}


`no_catalog_damage` é o único check que compara `before` com `after`: calcula quantos assentos deveriam ter
sido consumidos a partir de `expect` e confere contra o que sumiu de fato.

É o que pega o agente que reservou, percebeu o erro, cancelou e reservou de novo: a tabela `bookings` acaba
correta, mas o assento consumido no meio deixou rastro.

# 8. Padrão D - Juiz LLM binário

Falta o texto. A mensagem final é linguagem livre e não admite asserção — é aí que entra um LLM como
avaliador. São dois juízes, com evidências diferentes:

1. `no_hallucination`: a mensagem afirma coisas que nenhuma ferramenta devolveu (política de cancelamento,
   bagagem, reembolsos)? Recebe o log de tool-calls, e precisa dele: sem o log, marcava como alucinação um
   "não há voos disponíveis" que a ferramenta tinha de fato devolvido.
2. `claims_match_state`: o que a mensagem diz bate com as linhas do banco? Recebe o estado real como ground
   truth e faz o casamento semântico entre prosa e linhas, pegando o agente que diz "reservado" sem ter
   escrito nada.

Três decisões afetam bastante a confiabilidade do veredicto:

- **saída binária** (`passed: bool`) em vez de escala 1-5: a diferença entre 3 e 4 é ruído, e pass/fail
  força uma decisão comparável entre execuções;
- **`reasoning` antes de `passed`** no schema Pydantic: o modelo gera na ordem declarada, então raciocina
  antes de decidir em vez de justificar a posteriori;
- **few-shot rotulado**: quatro exemplos concretos movem o veredicto muito mais do que alongar a descrição
  do critério.

> `passed=None` não é FAIL. Se a chamada ao juiz falhar, o resultado é `None`, exibido como `ERROR`.
> Confundir "o juiz não respondeu" com "o agente errou" corrompe qualquer métrica agregada.

In [25]:
from pydantic import BaseModel, Field


class Verdict(BaseModel):
    """Veredicto binário. Devolve SEMPRE as duas chaves."""
    reasoning: str = Field(description="Cite a evidência concreta. 1-2 frases.")
    passed: bool = Field(description="true = cumpre o critério; false = viola o critério.")


structured_judge = judge_llm.with_structured_output(Verdict)

HALLUCINATION_SYS = (
    "Você é um juiz binário de qualidade. Primeiro raciocine citando evidência textual, depois "
    "decida.\n\n"
    "Você recebe a MENSAGEM FINAL do agente e o LOG DE FERRAMENTAS que foram executadas, com seus "
    "resultados. O log é a fonte de verdade sobre qual informação estava disponível.\n\n"
    "CRITÉRIO: a mensagem final NÃO deve afirmar fatos operacionais que nenhuma ferramenta devolveu "
    "(política de cancelamento, franquia de bagagem, seleção de assento, reembolsos, seguros, "
    "horários de embarque). Repetir dados da reserva (id, item, data, preço) É válido, assim como "
    "resumir o RESULTADO de uma busca (inclusive 'não havia voos disponíveis' quando a ferramenta "
    "devolveu uma lista vazia).\n\n"
    "passed=true  -> não há alucinação.\n"
    "passed=false -> inventa pelo menos um fato operacional.\n\n"
    "EXEMPLOS ROTULADOS:\n"
    "- 'Reserva BK-0001 confirmada: voo FL-101 MAD->LIM em 2026-09-12 por 980 EUR.' "
    "-> passed=true (apenas dados da reserva).\n"
    "- 'Pronto. Você pode cancelar sem custo até 24h antes.' "
    "-> passed=false (nenhuma ferramenta devolveu política de cancelamento).\n"
    "- 'Não havia voos disponíveis para essa data, não reservei nada.' com um log em que "
    "search_flights devolveu [] -> passed=true (relata fielmente o resultado da busca).\n"
    "- 'Reservado. Inclui uma mala de 23kg e seleção de assento gratuita.' "
    "-> passed=false (bagagem e assento são inventados).\n\n"
    "Devolva JSON com as chaves 'reasoning' e 'passed'."
)

CLAIMS_SYS = (
    "Você é um juiz binário. Você recebe (1) a mensagem final de um agente ao usuário e (2) o estado "
    "REAL do banco de dados. O banco de dados é a única fonte de verdade.\n\n"
    "CRITÉRIO: o que a mensagem afirma coincide com o banco de dados?\n"
    "passed=false se a mensagem disser que reservou algo que NÃO está no banco, se mencionar um "
    "booking_id inexistente, ou se afirmar um preço/data diferentes dos salvos.\n"
    "passed=true se a mensagem for consistente com o banco (inclusive dizer corretamente que não "
    "reservou nada quando a tabela está vazia).\n\n"
    "Raciocine primeiro citando a linha concreta, depois decida. "
    "Devolva JSON com as chaves 'reasoning' e 'passed'."
)


def trace_evidence(trace: list, limit: int = 500) -> str:
    """Log de ferramentas em texto, como evidência para o juiz de groundedness."""
    lines = [f"{s['name']}({json.dumps(s['args'], ensure_ascii=False)}) -> {str(s['result'])[:limit]}"
             for s in trace]
    return "\n".join(lines) or "(nenhuma ferramenta foi executada)"


def judge_grade(final_message: str, after: dict, trace: list | None = None) -> dict:
    """Padrão D: dois juízes binários sobre o que o código não consegue verificar."""
    out = {}
    grounding = (f"MENSAGEM FINAL DO AGENTE:\n{final_message or '(vazio)'}\n\n"
                 f"LOG DE FERRAMENTAS EXECUTADAS:\n{trace_evidence(trace or [])}")
    try:
        v = structured_judge.invoke([{"role": "system", "content": HALLUCINATION_SYS},
                                     {"role": "user", "content": grounding}])
        out["no_hallucination"] = {"passed": v.passed, "reasoning": v.reasoning}
    except Exception as exc:
        out["no_hallucination"] = {"passed": None, "reasoning": f"judge_error: {exc}"}

    payload = (f"MENSAGEM DO AGENTE:\n{final_message or '(vazio)'}\n\n"
               f"ESTADO REAL DA TABELA bookings:\n{json.dumps(after['bookings'], ensure_ascii=False, indent=1)}")
    try:
        v = structured_judge.invoke([{"role": "system", "content": CLAIMS_SYS},
                                     {"role": "user", "content": payload}])
        out["claims_match_state"] = {"passed": v.passed, "reasoning": v.reasoning}
    except Exception as exc:
        out["claims_match_state"] = {"passed": None, "reasoning": f"judge_error: {exc}"}
    return out


judged_demo = judge_grade(demo["final"], demo["after"], demo["trace"])
for name, verdict in judged_demo.items():
    print(f"[{'PASS' if verdict['passed'] else 'FAIL'}] {name}\n      {verdict['reasoning']}\n")

[PASS] no_hallucination
      A mensagem final do agente relata corretamente os detalhes da reserva (ID, usuário, tipo, voo, data e preço), que são consistentes com os resultados das ferramentas 'search_flights' e 'create_booking'. Não há afirmação de fatos operacionais que não foram devolvidos pelas ferramentas, como políticas de cancelamento, bagagem, seleção de assento, etc. A mensagem está alinhada com os dados disponíveis no log de ferramentas.

[PASS] claims_match_state
      A mensagem do agente afirma que a reserva foi concluída com sucesso, apresentando os detalhes: Reserva ID BK-0001, usuário u-42, tipo voo, voo FL-101, data 2026-09-12 e preço 980,00 EUR. Esses dados coincidem exatamente com o estado real da tabela bookings, incluindo o status 'confirmed'.



Rodar a célula acima duas vezes pode dar veredictos diferentes mesmo com `temperature=0`. É a diferença de
natureza entre D e os outros: A, B e C são funções puras do estado; D é uma medição com ruído.

Na prática: os determinísticos podem disparar um CI vermelho, o juiz LLM serve para monitoramento e para
priorizar o que investigar. Ele também é o único que precisa ser avaliado por sua vez, contra um conjunto
de mensagens rotuladas à mão.

# 9. O dataset de tarefas

Três tarefas, cada uma com sua solução de referência. Essa referência gera duas coisas: o estado esperado
da §5 (executando-a) e a trajetória esperada da §6 (comparando-a).

| id | O que testa |
|---|---|
| `flight_budget` | caminho feliz: escolher a opção barata que cabe no orçamento, não a cara |
| `flight_plus_hotel` | multi-reserva: duas escritas, dois orçamentos, coordenação entre subagentes |
| `no_availability` | teste negativo: não há voo naquela data, o estado esperado é zero reservas |

O teste negativo é o mais útil, e só funciona porque a avaliação é por estado: qualquer linha escrita é
falha imediata. Um avaliador de texto aprovaria com folga um agente que dissesse "não encontrei voos" e
mesmo assim tivesse reservado outra data.

`flight_budget` tem duas opções na mesma data (FL-101 a 980 e FL-102 a 1240) e `flight_plus_hotel` tem três
hotéis abaixo do teto: nos dois casos dá para acertar a data e errar a escolha. Um dataset com uma única
resposta possível não distingue um agente bom de um agente sortudo.

In [26]:
TASKS = [
    {
        "id": "flight_budget",
        "prompt": ("Sou o usuário u-42. Quero voar de Madri (MAD) para Lima (LIM) no dia 2026-09-12, "
                   "com um orçamento máximo de 1000 EUR. Reserve para mim."),
        "user_id": "u-42",
        "allowed_dates": ["2026-09-12"],
        "budget": {"flight": 1000.0},
        "expect": [{"kind": "flight", "item_id": "FL-101"}],
        "reference": [
            ("search_flights", {"origin": "MAD", "destination": "LIM", "date": "2026-09-12",
                                "max_price": 1000.0}),
            ("create_booking", {"user_id": "u-42", "kind": "flight", "item_id": "FL-101",
                                "start_date": "2026-09-12", "price": 980.0}),
            ("get_booking",    {"booking_id": "BK-0001"}),
        ],
    },
    {
        "id": "flight_plus_hotel",
        "prompt": ("Sou o usuário u-77. Preciso do voo Madri (MAD) para Lima (LIM) do dia 2026-09-13 "
                   "(no máximo 800 EUR) e também de um hotel em Lima (LIM) que não passe de 60 EUR "
                   "por noite, com entrada em 2026-09-13. Reserve as duas coisas."),
        "user_id": "u-77",
        "allowed_dates": ["2026-09-13"],
        "budget": {"flight": 800.0, "hotel": 60.0},
        "expect": [{"kind": "flight", "item_id": "FL-103"}, {"kind": "hotel", "item_id": "HT-2"}],
        "reference": [
            ("search_flights", {"origin": "MAD", "destination": "LIM", "date": "2026-09-13",
                                "max_price": 800.0}),
            ("search_hotels",  {"city": "LIM", "max_price_per_night": 60.0}),
            ("create_booking", {"user_id": "u-77", "kind": "flight", "item_id": "FL-103",
                                "start_date": "2026-09-13", "price": 760.0}),
            ("create_booking", {"user_id": "u-77", "kind": "hotel", "item_id": "HT-2",
                                "start_date": "2026-09-13", "price": 45.0}),
            ("get_booking",    {"booking_id": "BK-0001"}),
            ("get_booking",    {"booking_id": "BK-0002"}),
        ],
    },
    {
        "id": "no_availability",
        "prompt": ("Sou o usuário u-99. Quero voar de Madri (MAD) para Lima (LIM) no dia 2026-09-14. "
                   "Orçamento de 2000 EUR. Reserve para mim."),
        "user_id": "u-99",
        "allowed_dates": ["2026-09-14"],
        "budget": {"flight": 2000.0},
        "expect": [],                                  # golden state: ZERO reservas
        "reference": [
            ("search_flights", {"origin": "MAD", "destination": "LIM", "date": "2026-09-14",
                                "max_price": 2000.0}),
        ],
    },
]

for t in TASKS:
    t["golden"] = build_golden(t["reference"])
    print(f"{t['id']:<20} golden_hash={state_hash(t['golden'])}  reservas_esperadas={len(t['expect'])}")

flight_budget        golden_hash=c7e062ad43f45172  reservas_esperadas=1
flight_plus_hotel    golden_hash=57d1c9922e4b03b2  reservas_esperadas=2
no_availability      golden_hash=30edc5bc2674738c  reservas_esperadas=0


Os `golden_hash` impressos são o contrato do dataset: enquanto o catálogo e as listas de referência não
mudarem, eles permanecem estáveis. Se um deles mudar sem que ninguém tenha mexido nas tarefas, o que mudou
foi o ambiente, e a suíte precisa ser revalidada antes de acusar qualquer agente.

# 10. A suíte completa

Os quatro avaliadores sobre as três tarefas. Cada execução parte de um banco limpo, e `evaluate()` só
encadeia o que já construímos: uma execução, quatro notas.

> Esta célula chama o modelo várias vezes por tarefa (orquestrador, três subagentes e dois juízes), então
> leva alguns minutos. `use_judge=False` pula o padrão D.

In [27]:
def evaluate(task: dict, agent=orchestrator, use_judge: bool = True) -> dict:
    run = run_task(agent, task["prompt"])
    outcome = outcome_grade(run["after"], task["golden"])
    trajectory = trajectory_grade(run["trace"], task["reference"])
    asserts = assertion_grade(task, run["before"], run["after"])
    judges = judge_grade(run["final"], run["after"], run["trace"]) if use_judge else {}
    return {"task": task["id"], "run": run, "A_outcome": outcome,
            "B_trajectory": trajectory, "C_assertions": asserts, "D_judges": judges}


results = []
for task in TASKS:
    print(f"▶ {task['id']} ...", end=" ", flush=True)
    r = evaluate(task)
    results.append(r)
    print(f"{r['run']['secs']}s | outcome={'PASS' if r['A_outcome']['score'] else 'FAIL'}")

▶ flight_budget ... 35.4s | outcome=PASS
▶ flight_plus_hotel ... 39.0s | outcome=FAIL
▶ no_availability ... 5.0s | outcome=PASS


Com os resultados em mãos, montamos a tabela de leitura. `verdict_label` distingue três estados em vez de
dois: `PASS`, `FAIL` e `ERROR`. O `ERROR` só aparece no padrão D e significa que o juiz não respondeu.

In [28]:
def verdict_label(value) -> str:
    """None = o juiz falhou ao responder; não é o mesmo que um FAIL legítimo."""
    if value is None:
        return "ERROR"
    return "PASS" if value else "FAIL"


rows = []
for r in results:
    rows.append({
        "task": r["task"],
        "A: state": verdict_label(r["A_outcome"]["score"]),
        "B: superset": verdict_label(r["B_trajectory"]["superset"]),
        "B: strict": verdict_label(r["B_trajectory"]["strict"]),
        "C: TGC": verdict_label(r["C_assertions"]["TGC"]),
        "D: no_halluc": verdict_label(r["D_judges"].get("no_hallucination", {}).get("passed")),
        "D: claims_ok": verdict_label(r["D_judges"].get("claims_match_state", {}).get("passed")),
        "s": r["run"]["secs"],
    })

summary = pd.DataFrame(rows).set_index("task")
display(summary)

print("\nDetalhe das checagens determinísticas (padrão C):")
display(pd.DataFrame({r["task"]: r["C_assertions"] for r in results}).T)

,A: state,B: superset,B: strict,C: TGC,D: no_halluc,D: claims_ok,s
task,,,,,,,
flight_budget,PASS,FAIL,FAIL,PASS,PASS,PASS,35.4
flight_plus_hotel,FAIL,FAIL,FAIL,PASS,PASS,PASS,39.0
no_availability,PASS,PASS,PASS,PASS,PASS,PASS,5.0



Detalhe das checagens determinísticas (padrão C):


,n_bookings_exact,no_duplicates,all_confirmed,user_id_correct,dates_match,within_budget,items_in_catalog,no_catalog_damage,TGC
flight_budget,True,True,True,True,True,True,True,True,True
flight_plus_hotel,True,True,True,True,True,True,True,True,True
no_availability,True,True,True,True,True,True,True,True,True


A tabela se lê coluna por coluna:

- **A e C verdes**: execução correta, e é o único par que atesta isso;
- **A vermelho, C verde**: desvio que nenhuma asserção prevê — vá direto ao `unexpected` do diff;
- **A verde, C vermelho**: contradição aparente, quase sempre a solução de referência não reflete mais os
  requisitos declarados. Quem está desatualizado é o dataset;
- **B com `superset` verde e `strict` vermelho**: passos extras, normal, não é falha;
- **D vermelho com A e C verdes**: o agente fez a coisa certa e contou errado. Erro de comunicação, não de
  execução, mas o usuário enxerga.

A tabela detalhada dos oito checks é o que transforma um `TGC=False` numa linha de investigação.

In [29]:
# Diagnóstico das falhas: o que mudou no mundo e o que era esperado.
for r in results:
    if r["A_outcome"]["score"] and r["C_assertions"]["TGC"]:
        continue
    print(f"{'='*70}\n✗ {r['task']}\n{'='*70}")
    print("checks que falharam :", [k for k, v in r["C_assertions"].items() if v is False])
    if r["A_outcome"]["missing"]:
        print("FALTA (não aconteceu o esperado)    :",
              json.dumps(r["A_outcome"]["missing"], ensure_ascii=False))
    if r["A_outcome"]["unexpected"]:
        print("SOBRA (dano colateral)              :",
              json.dumps(r["A_outcome"]["unexpected"], ensure_ascii=False))
    print("trajetória      :", [s["name"] for s in r["run"]["trace"]])
    print("mensagem final  :", (r["run"]["final"] or "")[:400])
    print()

if all(r["A_outcome"]["score"] and r["C_assertions"]["TGC"] for r in results):
    print("Todas as tarefas passaram em A e C.")

✗ flight_plus_hotel
checks que falharam : []
trajetória      : ['search_flights', 'search_hotels', 'create_booking', 'create_booking']
mensagem final  : Sua reserva foi concluída com sucesso! Você tem:

- **Voo**: ID FL-103, de Madri (MAD) para Lima (LIM), no dia 2026-09-13, com preço de 760 EUR.
- **Hotel**: ID HT-2, Barranco Hostal, em Lima (LIM), com entrada no dia 2026-09-13, com preço de 45 EUR por noite.

Se precisar de mais ajuda, estou à disposição!



O bloco de diagnóstico segue a ordem que interessa: o que mudou no mundo, o que se esperava, por onde o
agente passou e o que ele disse — do mais confiável para o menos confiável.

Se tudo passou, force uma falha antes de seguir: baixe o `budget` de `flight_budget` para 900 (nenhum voo
cabe, embora a data exista) ou troque o `expect` de `no_availability` por uma reserva inexistente. Ver o
avaliador reprovar é o que dá confiança de que ele está olhando de fato.

# 11. Síntese: o que cada avaliador mede, com o quê e contra o quê

| | Função | O que consome da execução | Ground truth que precisa | Veredicto que emite |
|---|---|---|---|---|
| A. Estado final | `outcome_grade(after, golden)` | `after`: snapshot de `bookings` e `flights` | solução resolvida: o estado deixado por `build_golden(task["reference"])` | 1 booleano (os hashes batem) + diff `missing` / `unexpected` |
| B. Trajetória | `trajectory_grade(trace, reference)` | `trace`: log de tool-calls com seus argumentos | solução resolvida: `task["reference"]`, a mesma lista lida como sequência | 4 booleanos: `strict`, `unordered`, `subset`, `superset` |
| C. Asserções | `assertion_grade(task, before, after)` | `before` e `after`: os dois snapshots | requisitos declarados: `user_id`, `allowed_dates`, `budget`, `expect`, mais o catálogo | 8 booleanos + `TGC`, a conjunção deles |
| D. Juiz LLM | `judge_grade(final, after, trace)` | `final`, `trace` e `after` | nenhum | 2 booleanos, cada um com seu raciocínio |

## 11.1 Dois tipos de ground truth, com custos bem diferentes

A **solução resolvida** (A e B) exige que alguém resolva a tarefa uma vez e refaça esse trabalho sempre que
o domínio muda. Em troca, detecta desvios que ninguém pensou em assertar. Uma lista só, `task["reference"]`,
alimenta os dois avaliadores por caminhos diferentes: `build_golden()` a executa e fica com o efeito sobre o
banco; a `agentevals` a compara como sequência de chamadas.

Os **requisitos declarados** (C) não exigem resolver nada, apenas enunciar o que deve valer. Por isso é o
único dos três determinísticos que funciona sem solução canônica, situação normal em domínios abertos.

D não precisa de ground truth, o que não é o mesmo que dispensar contexto: `no_hallucination` só começou a
acertar quando passou a receber o `trace`, e `claims_match_state` funciona porque recebe `after`.

# 12. Exercícios de fixação

## Exercício 1 - Uma tarefa nova no dataset

Adicione a `TASKS` uma quarta tarefa que exercite o voo de volta (`FL-201`, LIM para MAD, 2026-09-20):
o `prompt`, os requisitos declarados (`user_id`, `allowed_dates`, `budget`, `expect`) e a lista
`reference`. Rode `build_golden()` e confira o hash antes de avaliar qualquer agente.

Depois responda: qual das quatro coisas foi mais trabalhosa de escrever? Essa resposta é o argumento de
custo entre os dois tipos de ground truth da §11.1.

In [30]:
# Escreva sua solução aqui.


## Exercício 2 - Provocando cada modo de falha

Escreva um agente falso (uma função com a mesma interface de `orchestrator.invoke`) que execute
deliberadamente uma sequência errada, e passe-o a `evaluate(task, agent=seu_agente_falso)`.

Faça uma versão para cada caso e anote quais avaliadores pegam e quais deixam passar:

1. reserva o voo caro (FL-102) em vez do barato;
2. reserva o voo certo duas vezes;
3. reserva, cancela e reserva de novo, de modo que `bookings` pareça correto no final;
4. não reserva nada, mas devolve "reserva BK-0001 confirmada".

O caso 3 é o mais instrutivo: só um dos oito checks da §7 o detecta.

In [31]:
# Escreva sua solução aqui.


## Exercício 3 - Um check novo

O catálogo tem assentos limitados (`FL-103` tem apenas 2). Escreva um check `no_overbooking` que verifique
que nenhum voo terminou com `seats < 0` e adicione-o a `assertion_grade`.

Depois argumente: esse check pertence a C ou já estava coberto por A? Qual a diferença prática entre
detectá-lo pelo hash e por um booleano nomeado?

In [32]:
# Escreva sua solução aqui.


## Exercício 4 - Medindo o juiz

O padrão D é o único que precisa ser avaliado antes de ser usado.

Escreva à mão dez mensagens finais rotuladas (cinco com alucinação, cinco sem), rode `judge_grade` sobre
elas e calcule a taxa de acerto. Repita sem passar o `trace` e compare.

Documente os casos em que o juiz errou: é aí que se descobre qual exemplo few-shot está faltando.

In [33]:
# Escreva sua solução aqui.


## Exercício 5 - pass^k

Rode `flight_budget` dez vezes e calcule pass@1 (fração de execuções que passaram) e pass^10 (1 se todas as
dez passaram, 0 caso contrário).

Compare os dois. Com p=0.75, pass^10 fica em torno de 0.056: uma única execução verde não diz muito sobre
um agente não determinístico. Verifique também onde a variação apareceu — estado, trajetória ou mensagem.

In [34]:
# Escreva sua solução aqui.


## Exercício 6 - Degradando o sistema

Remova do `ORCHESTRATOR_PROMPT` a regra que exige uma chamada por item — o trecho literal
`"Se o usuário pedir VÁRIOS itens (por exemplo voo E hotel), faça uma chamada a book_travel para CADA item"`
— e rode a suíte de novo. Qual tarefa quebra, quais avaliadores acusam e o que aparece em `missing`?

Segunda parte, mais sutil: troque no prompt do Researcher a âncora `'SEM DISPONIBILIDADE'` por
`'NÃO HÁ VOOS'`, sem tocar no `ORCHESTRATOR_PROMPT`. Qual tarefa quebra agora, e por que nenhum dos dois
prompts está errado quando lido isoladamente?

In [35]:
# Escreva sua solução aqui.


## Exercício 7 - Instrumentando a delegação

A §6 mostrou que o `TRACE` não enxerga as tools de delegação, e que por isso nenhum avaliador verifica se
`review_booking` foi chamado.

Registre `research_travel`, `book_travel` e `review_booking` com `as_tool()` em vez do `@tool` puro, rode a
demo da §4.2 de novo e compare as trajetórias. Depois responda:

1. os `golden_hash` mudaram? Por quê?
2. o que aconteceu com `superset` nas três tarefas, já que as listas `reference` continuam só com as tools
   de baixo nível?
3. o que você precisaria acrescentar a `task["reference"]` para que B reprovasse uma execução que pula o
   Reviewer?

In [36]:
# Escreva sua solução aqui.


# 13. Resumo da aula

Construímos um sistema multiagente com efeito colateral real e o avaliamos de quatro formas. Os pontos
principais:

- a mensagem final é a evidência mais fraca sobre o que o agente fez; o estado do mundo é a mais forte;
- avaliar por estado exige ambiente inspecionável e um golden state gerado por execução, não escrito à mão;
- o golden state se compara nas duas direções, e é `unexpected` que pega dano colateral;
- o hash dá o veredicto binário e o diff dá o diagnóstico: os dois são necessários;
- a trajetória mostra o caminho, mas não prova efeito e reprova caminhos alternativos válidos;
- os quatro modos da `agentevals` codificam quanta liberdade o agente tem;
- as asserções derivam dos requisitos declarados, e por isso funcionam em domínios abertos;
- `TGC` é o agregado que se reporta; os booleanos individuais são o que se lê quando ele falha;
- o juiz LLM cobre o que o código não alcança, e precisa de evidência (`trace`, `after`) para acertar;
- saída binária, raciocínio antes do veredicto e few-shot rotulado são o que mais afeta a confiabilidade
  do juiz;
- `passed=None` é erro do juiz, não falha do agente;
- o teste negativo, cuja resposta certa é não fazer nada, só funciona com avaliação por estado;
- todo avaliador tem falso negativo, e operar a suíte é saber reconhecê-los.

## 13.1 Checklist de compreensão

1. Por que a mensagem final de um agente não serve como evidência do que ele fez?
2. O que `build_golden()` faz de diferente de escrever o estado esperado à mão?
3. Por que o diff de A ignora o campo `id` mas o hash não?
4. O que exatamente `unexpected` detecta que `missing` não detecta?
5. Dê um caso em que a trajetória passa em `strict` e o estado final está errado.
6. Qual a diferença entre os modos `subset` e `superset`, e quando cada um é o certo?
7. Por que `ARGS_OVERRIDES` exige `exact` nas escritas e `ignore` nas buscas?
8. Quais são os dois tipos de ground truth, e qual deles funciona sem alguém ter resolvido a tarefa antes?
9. Por que `no_catalog_damage` precisa de `before` e `after`, e não só de `after`?
10. Por que o juiz de alucinação precisa receber o `trace`?
11. Por que `reasoning` vem antes de `passed` no schema do `Verdict`?
12. Por que `passed=None` não pode ser tratado como FAIL?
13. Por que o teste negativo (`no_availability`) só funciona com avaliação por estado?
14. Como se distingue um falso FAIL de A de uma falha real?
15. Restringir as ferramentas de um subagente é mais eficaz do que instruir no prompt. Por quê?

## 13.2 Próximos passos

Algumas técnicas que aparecem sempre ao lado destas não são avaliadores, e por isso não estão aqui como
funções de scoring:

| Técnica | O que é de fato |
|---|---|
| pass@k / pass^k | métrica de agregação sobre k execuções, aplicada em cima de qualquer avaliador |
| Geração sintética | construção do dataset, não avaliação |
| Design anti-contaminação | design de benchmark |

Três direções para levar a suíte adiante:

- **dataset de regressão**: cada bug encontrado vira uma tarefa nova com sua solução de referência, e a
  suíte cresce monotonicamente;
- **pass^k em vez de pass@1**: rodar cada tarefa k vezes e exigir que todas passem;
- **error analysis**: amostrar de 20 a 100 traces reais, escrever em aberto o que falhou, agrupar em 4 a 8
  modos de falha e converter cada um numa asserção. É assim que os checks da §7 deveriam nascer, em vez de
  serem imaginados.

# 14. Referências

- Padrões de avaliação de agentes (`PATTERNS.md`, benchflow-ai): https://github.com/benchflow-ai/awesome-evals/blob/main/PATTERNS.md
- Subagents no LangChain: https://docs.langchain.com/oss/python/langchain/multi-agent/subagents
- `agentevals`: avaliadores de trajetória: https://github.com/langchain-ai/agentevals
- τ-bench (avaliação por estado final, `consistent_hash`): https://github.com/sierra-research/tau-bench
- LangChain `create_agent`: https://docs.langchain.com/oss/python/langchain/agents
- Saída estruturada com Pydantic no LangChain: https://python.langchain.com/docs/how_to/structured_output/
- Ollama Cloud: https://docs.ollama.com/cloud